In [ ]:
import numpy as np
import pandas as pd
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "Source"))

from fractions import Fraction
from Calc_PerGraph import (
    enumerate_blocking_sequences,
    compute_occupied_count_distributions
)

NameError: name 'sys' is not defined

In [ ]:
n = 8

edges = [
    (1, 2), (1, 3), (2, 3),  # Primer triángulo
    (2, 4), (2, 8), (4, 8),  # Segundo triángulo
    (2, 7), (7, 5), (5, 6), (6, 2)  # Ciclo de longitud 4
]

A = [[0] * n for _ in range(n)]

for u, v in edges:
    A[u-1][v-1] = 1
    A[v-1][u-1] = 1

In [ ]:
def fraction_latex(x, max_denominator=10**6):
    """
    Convierte una probabilidad numérica a una fracción LaTeX.
    """
    f = Fraction(float(x)).limit_denominator(max_denominator)

    if f.denominator == 1:
        return f"${f.numerator}$"

    return rf"$\frac{{{f.numerator}}}{{{f.denominator}}}$"


def dominance_table(adjacency_matrix, tolerance=1e-9):
    """
    Construye la tabla de distribuciones acumuladas y determina
    si se cumplen D1, D2, ambas o ninguna.

    D1: Sequential domina Static.
    D2: Static domina Sequential.

    Returns:
        df: DataFrame con las distribuciones acumuladas.
        dominance: clasificación de dominación.
    """

    # Enumerar las secuencias y calcular las distribuciones.
    sequences = enumerate_blocking_sequences(adjacency_matrix)

    stat_dist, seq_dist = compute_occupied_count_distributions(sequences)

    # Soporte conjunto.
    ks = sorted(set(stat_dist) | set(seq_dist))

    # Distribuciones acumuladas.
    cumulative_stat = np.cumsum([
        stat_dist.get(k, 0.0) for k in ks
    ])

    cumulative_seq = np.cumsum([
        seq_dist.get(k, 0.0) for k in ks
    ])

    # Verificar dominación estocástica.
    d1 = all(
        cumulative_stat[i] >= cumulative_seq[i] - tolerance
        for i in range(len(ks))
    )

    d2 = all(
        cumulative_seq[i] >= cumulative_stat[i] - tolerance
        for i in range(len(ks))
    )

    # Clasificación.
    if d1 and d2:
        dominance = "Ambos"
    elif d1:
        dominance = "D_1 (Seq domina Stat)"
    elif d2:
        dominance = "D_2 (Stat domina Seq)"
    else:
        dominance = "Ninguno"

    # Construir tabla.
    df = pd.DataFrame({
        "$k$": ks,
        r"$\mathbb{P}[N_G^{\mathrm{stat}}\leq k]$": [
            fraction_latex(x) for x in cumulative_stat
        ],
        r"$\mathbb{P}[N_G^{\mathrm{seq}}\leq k]$": [
            fraction_latex(x) for x in cumulative_seq
        ],
        "Dominancia": [dominance] * len(ks)
    })

    return df, dominance

In [ ]:
df, dominance = dominance_table(A)

display(df)

print("Resultado de dominación:", dominance)

In [ ]:
latex_table = df.to_latex(
    index=False,
    escape=False,
    column_format="cccc"
)

print(latex_table)